# 4 Co-creating an image-analysis function with an LLM
In this module, programmer and language model work together to write `measure_near_cilia(green_image, red_image, threshold, radius)` to segment the cilia, dilate the mask, and return the protein-channel intensity sum, mask area, and mean intensity. The two images, **Image_34362.tif** and **Image_34363.tif**, are included in this directory.

Section 4.1 provides a self-contained base case for the measurement. The LLM contributes proposals and code; you contribute the intended measurement, design decisions, evidence, and judgement. A plausible answer is a proposal to evaluate, not a completed analysis.

We use three complementary pedagogical tools:

- **Copy–modify–create** guides the progression: copy established processing steps or a draft, modify them for the agreed task, then create a reusable solution by combining those ideas.
- **Predict–code–reflect** guides each activity: anticipate the result, develop and run code, then reflect on the evidence. During planning, a written artefact takes the place of code: predict, co-create the artefact, then reflect on its quality.
- **Review–refine** guides the collaboration with the LLM. Start with a prompt asking for an initial proposal or presenting a nearly complete draft. Review the response against the task and previously accepted artefacts. Give specific feedback and decisions in a follow-up prompt, then inspect the revision. Repeat when needed until the team can justify accepting the artefact.

The planning assignments are co-created too. Section 4.2 introduces a shared prompting and review–refine method used throughout. Acceptance is based on reasons and evidence rather than on completing a fixed number of exchanges.

The sequence is: requirement analysis → design choices → code specification → quality conditions and checks → assembled prompt → proposed implementation → execution and evaluation. The team and LLM co-create each stage, but the team decides what is accepted. A change to an earlier decision may require updating later artefacts. The full workflow turns the base-case calculation in section 4.1 into a reusable function using the same local images.

Keep written artefacts, prompts, responses, and revisions in this notebook so a colleague can understand how the result was obtained. Reflection concerns the existing results and decisions; further revision belongs in the activity before its Reflect prompt.

## A “brain-rot” warning: keep mental ownership
Working with an LLM can distance your ideas from the solution. The model can produce decisions and code faster than you can understand them, making it tempting to accept plausible answers and become a passive reviewer. In this course, “brain-rot” names that risk of intellectual laziness: outsourcing the thinking as well as the typing.

Responsibility for the quality of the solution remains entirely yours as the programmer. The LLM's confidence, a convincing explanation, and code that runs do not establish that the result answers your question. You need conscious steps to become the mental owner of the result:

- Form your own prediction and identify the important requirements before accepting a proposal.
- Review the model's choices against your intended measurement, and ask about assumptions you cannot justify.
- Trace each processing step and account for it in your own words, including what the inputs, parameters, and outputs mean.
- Check expected results independently and examine failures rather than relying on the model's claim that its code is correct.
- Judge whether you could adapt the solution to a changed requirement or diagnose a wrong result. If not, identify the part you still need to understand.

The purpose of co-creation is to use the LLM's contributions while retaining your own judgement. Review–refine and predict–code–reflect are deliberate ways to stay engaged and take ownership of the accepted solution.

## 4.1 Base case: protein intensity near cilia
The base case is a direct calculation of protein-channel intensity near cilia. `Image_34362.tif` shows the cilia mainly in green, and `Image_34363.tif` shows the protein of interest mainly in red. The images represent corresponding pixel positions; one image's mask can therefore select pixels in the other.

An RGB image is a table indexed by row, column, and colour channel. Channels 0, 1, and 2 are red, green, and blue. The expression `cilia[:,:,1]` selects all rows and columns of the green channel, giving a two-dimensional array. `protein[:,:,0]` selects the red channel.

Open this notebook with `4_LLM` as the working directory. The code reads the two local files, selects the channels, and displays the images. NumPy handles arrays, scikit-image provides image reading and morphology, and Matplotlib displays figures.

**Predict:** Which channel belongs to each measurement, and should these images have the same shape?

In [ ]:
from skimage import io, morphology
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['image.cmap'] = 'gray'

# Read the two images included alongside this notebook.
cilia = io.imread('Image_34362.tif')
protein = io.imread('Image_34363.tif')
green = cilia[:,:,1]
red = protein[:,:,0]
plt.imshow(cilia)
plt.title('Cilia')
plt.show()
plt.imshow(protein)
plt.title('Protein')
plt.show()


**Reflect:** Do the displayed colours agree with the intended cilia and protein channels? Why must the channels have matching shapes for one mask to select protein pixels?

The direct calculation has three steps:

1. **Segment:** `green > chosenThreshold` creates a Boolean mask. A pixel is True only when its green intensity is strictly greater than the threshold.
2. **Dilate:** expand the selected pixels with a disk of the chosen radius, measured in pixels. The expanded mask includes the original cilia pixels and their neighbourhood; expansion stops at the image boundary.
3. **Measure:** multiply red intensities by the expanded mask, retaining selected values and setting the others to zero. Their sum is the total intensity. The number of True pixels is the mask area. Dividing the sum by the area gives mean intensity inside the mask. If no pixels are selected, the mean is undefined and recorded as `np.nan`.

Threshold 75 and radius 20 are illustrative choices, not automatically optimal. This base-case calculation supplies comparison values for the eventual function. The function will accept those choices as arguments so it can be reused without editing its body.

**Predict:** What do the three reported quantities represent? Does the expanded mask include the cilia, only their surroundings, or both?

In [ ]:
# Direct base-case calculation with explicit threshold and radius choices.
chosenThreshold = 75
chosenRadius = 20
segm = green > chosenThreshold
msk = morphology.dilation(segm, morphology.disk(chosenRadius))
redMsk = red.astype(float) * msk
referenceTotal = redMsk.sum()
referenceArea = msk.sum()
if referenceArea == 0:
    referenceMean = np.nan
else:
    referenceMean = referenceTotal / referenceArea
print('Reference intensity sum:', referenceTotal)
print('Reference mask area (pixels):', referenceArea)
print('Reference mean intensity:', referenceMean)
plt.imshow(redMsk)
plt.title('Protein intensities inside the expanded cilia mask')
plt.show()


**Reflect:** What does the sum measure, what does the area count, and how is the mean obtained? Which aspects depend on the threshold and radius, and why is this fluorescence measurement not automatically an absolute amount of protein?

## 4.2 Requirement analysis
Requirement analysis describes the problem from the user's point of view before choosing how to implement it. It establishes who needs the result, what question it answers, what information is available, and what a satisfactory output means.

Useful requirements identify inputs, outputs, units, constraints, and the boundaries of the task. They also cover unusual situations, such as a mask that contains no pixels. For this measurement, “near the cilia” needs a clear meaning: including the cilia themselves gives a different measurement from including only their surroundings.

A requirement states a need, such as reusing the measurement with another image pair. A design choice states a way to meet it, such as passing images into a function rather than loading filenames inside it. Keeping these separate leaves room to compare implementations without changing the scientific question.

An LLM can help identify missing requirements or ambiguous wording. The programmer and intended user remain responsible for deciding which suggestions describe the actual task.

### Co-create, review, and refine
The same collaboration method applies to all the following assignments. Begin either with a request for an initial proposal based on a few requirements, or with a nearly complete draft for the LLM to review. Include the base-case task and your accepted earlier decisions; the model is not assumed to have this notebook.

For example, an initial prompt could be:

```text
We are co-creating a reusable function that thresholds a cilia image, dilates its mask, and returns protein-channel intensity sum, mask area, and mean. Help us develop [the artefact for this assignment]. Here are our requirements and accepted decisions: [include the relevant context]. Propose an initial version and identify unresolved choices. Do not produce later-stage artefacts yet.
```

Alternatively, supply your draft and ask which parts meet the agreed requirements, which remain ambiguous, and what needs refinement.

Review the response against the current task and accepted earlier work. Discuss at least one specific ambiguity, omission, improvement, or decision needing justification. A follow-up prompt could be:

```text
Our review found [specific issue or decision]. We intend [behaviour or choice] because [reason or independently checked evidence]. Revise the artefact accordingly, preserving the other accepted decisions. Identify any consequences for earlier work.
```

Inspect the revision and continue when needed until the team can justify accepting it. Keep the accepted artefact and enough of the prompts, responses, and review decisions to account for its development. The clearly marked solution area below each assignment is yours to update; choose a form that suits your work. For code assignments, use the code cell and preserve your agreed function interface. Reflect on the accepted work after the activity.

**Assignment 4a:** Co-create and agree on a requirement analysis for the base-case measurement.

**Predict:** Which ambiguities could lead to an implementation that runs but answers a different question?

**Our requirements:** \<Add your solution here\>

**Reflect:** Does the requirement analysis clearly describe the base-case measurement? Could someone interpret “near”, the measurement units, or the empty-mask result differently? What evidence would establish that the user’s need was met? How did the artefact change through review and refinement, and what justified accepting the revised version?

## 4.3 Design choices
Design describes how a solution will meet the requirements. Several designs may satisfy the same need, but differ in readability, numerical reliability, reuse, and ease of changing the code later.

For this task, the existing building blocks are thresholding, disk dilation, and masked intensity summing. Design connects those steps and decides which values are parameters, how results are returned, and which responsibilities belong outside the function. For example, keeping plotting outside a measurement function makes the function usable when no display is needed.

Numerical representation is also a design choice. Converting intensities to float before arithmetic avoids the limited range of small integer types. Empty-mask handling might return an undefined mean or raise an error; the choice must agree with the requirements.

A short design record states the choice, its reason, and an alternative. It helps a future reader understand why the solution has its present structure. The simplest design that meets the requirements is often easiest to inspect and maintain.

**Assignment 4b:** Co-create a design that meets your accepted requirements, including the reasons for your choices.

**Predict:** Which choices will make this function easier to reuse with another image pair or another radius?

**Our design choices:** \<Add your solution here\>

**Reflect:** Which choices follow directly from requirements, and which are replaceable design decisions? How do they affect readability, reuse, and future changes? Is any extra complexity unsupported by the task? How did the artefact change through review and refinement, and what justified accepting the revised version?

## 4.4 Code specification
A code specification turns the requirements and design into a precise description that another programmer can implement. It acts as a contract between the caller and the function: what inputs are supported, what the function does, and what results or errors the caller can expect.

The interface consists of the function name, parameters, and returned values. The reusable measurement will use the interface `measure_near_cilia(green_image, red_image, threshold, radius)`. A specification also states the processing steps, comparison rules, units, return order, side effects, and behaviour in exceptional cases.

Small details matter. `>` and `>=` differ for pixels exactly at the threshold, and a radius in pixels is not a radius in micrometres. An empty-mask policy determines whether the caller receives an undefined value, a zero, or an error.

Input assumptions are called preconditions. A specification distinguishes preconditions that the caller must satisfy from invalid inputs that the function must detect. This distinction makes its responsibilities clear and helps determine what should be tested.

**Assignment 4c:** Co-create a precise code specification based on your accepted requirements and design.

**Predict:** Would two implementers given only your specification select the same pixels and return the same three quantities?

**Our code specification:** \<Add your solution here\>

**Reflect:** Is the specification precise enough to judge an implementation? Where does it leave freedom, and where would different interpretations change the measurement? Does every required behaviour trace back to a requirement or a stated design decision? How did the artefact change through review and refinement, and what justified accepting the revised version?

## 4.5 Quality conditions before code generation
Quality conditions state the evidence needed to accept a solution. They come from the requirements and specification, rather than from whichever implementation the LLM happens to propose.

Some conditions have numerical answers: a known mask area, a hand-calculated intensity sum, or a particular result for an empty mask. Others concern behaviour, such as leaving inputs unchanged, or qualities that require inspection, such as readable names, useful documentation, and independence from notebook variables.

A test case combines an input, an action, and an expected result. Tiny arrays are useful because their answers can be calculated by hand. They are test fixtures, not substitutes for the local microscopy images. Threshold equality, image boundaries, large intensities, different shapes, and invalid inputs expose errors that a plausible-looking microscopy result may conceal.

Expected results must be independent of the function under test. Comparing a function with itself does not establish correctness. The expected empty-mask result must match the agreed specification. Passing tests supports only the conditions they cover; it does not establish biological validity or complete freedom from bugs.

**Assignment 4d:** Co-create quality conditions for your specification and verify expected numerical results independently.

**Predict:** Which plausible implementation mistakes would a microscopy image alone fail to reveal?

**Our quality conditions:** \<Add your solution here\>

**Reflect:** Which conditions have independent expected values, and which require inspection or judgement? What errors could still pass the planned checks? How do the quality conditions cover the requirements without simply repeating the proposed implementation? How did the artefact change through review and refinement, and what justified accepting the revised version?

### Turn conditions into simple checks
An automated check makes a quality condition executable. An `assert` states something that must be true; if it is false, Python raises `AssertionError`. For example, `assert area == 1` checks a known area. A descriptive message helps identify what failed.

`np.isnan(mean)` checks an undefined numerical result; `mean == np.nan` does not work for this purpose. `np.testing.assert_allclose` compares floating-point results within a tolerance. Checks can be grouped in a function: defining it records the checks, while calling it executes them.

`np.zeros((3,3))` makes a small array of zeros, and `np.full((3,3), 200)` makes one filled with 200. Setting `[1,1]` changes its centre pixel. For a check that expects an error, `try` attempts a call and `except ValueError` catches that specific error; the `else` branch runs if no error occurred. The worked solution uses this pattern for invalid inputs.

Here the checks are defined before code generation but called only after the implementation is pasted in section 4.7. An LLM may suggest additional checks, but their expected values still need independent scrutiny.

**Assignment 4e:** Co-create executable checks for your accepted quality conditions. Paste the accepted definitions below without calling them yet.

**Predict:** For the 3 × 3 arrays, what are sum, area, and mean at radius 0? At radius 1, how many neighbours does the disk include?

In [ ]:
# Our checks: <Add your solution here>


**Reflect:** Do the expected values follow from the arrays and specification? Which wrong implementations would fail these checks? Are the checks independent enough to be useful even if the LLM misunderstands the task? How did the artefact change through review and refinement, and what justified accepting the revised version?

## 4.6 Assemble the prompt
A prompt communicates the agreed task to the LLM. In this workflow, it packages decisions already made: requirements, design, code specification, and quality conditions. This reduces the number of missing decisions the model must guess.

A self-contained prompt includes the function interface, relevant processing steps, input assumptions, expected outcomes, and requested response format. The model cannot be assumed to know this notebook or have access to its image files. The prompt can request that ambiguities be identified before implementation rather than silently resolved.

Requesting implementation and tests in separate Python blocks makes the response easier to paste and inspect. The organisation and names of the checks should follow your agreed specification; their definitions are executed before the checks are run. A brief account of assumptions and which checks were actually executed helps distinguish a proposal from verified evidence.

The LLM may help improve the prompt, but the team decides whether a suggested change preserves the agreed task. Saving the complete prompt and subsequent response makes the collaboration traceable.

**Assignment 4f:** Co-create a self-contained implementation prompt from your accepted planning artefacts. Save it below and use it in Assignment 4g.

**Predict:** What would the LLM have to guess if you omitted the return order, radius units, or empty-mask policy?

**Our complete prompt:** \<Add your solution here\>

**Reflect:** Is the prompt self-contained? Could the LLM implement the specified function without inventing missing decisions? Do its requested output and checks support evaluation and later reuse? How did the artefact change through review and refinement, and what justified accepting the revised version?

## 4.7 Implement and verify
The LLM proposes an implementation; you decide whether it meets the specification. Independent checks and comparison with the base case provide evidence. If a check fails, use its expected and observed results in a review–refine exchange rather than silently changing the requirement.

Paste Python without Markdown fences. Use your agreed names, define the implementation before running its checks, and keep a brief record of the response and revisions. Restarting the kernel and running the notebook in order checks for hidden dependencies.

**Assignment 4g:** Obtain and inspect the implementation against the quality conditions agreed in Assignment 4d. Paste it below, run the checks developed in 4e, and assess any conditions that require inspection or judgement. Compare the three measurements with the base case and call the function with a different radius. Use the evidence to decide which quality conditions are met and refine the solution with the LLM where needed.

**Predict:** Which quality conditions from 4d do you expect the implementation to satisfy, and what evidence will demonstrate each one?

In [ ]:
# Our implementation and verification: <Add your solution here>


**Reflect:** Which quality conditions from 4d are supported by the checks, comparisons, and inspection, and which remain unmet or uncertain? Does that evidence justify accepting the implementation? Could you account for the processing steps and reuse the function without editing its body?

**Our generation and revision record:** \<Add your solution here\>

## 4.8 Evaluate the co-created result
Quality concerns the whole process: requirements, design, specification, checks, prompt, and implementation. A working program alone does not establish that the intended measurement is correct.

Reuse means applying the function in another supported situation; maintenance means understanding and changing it without accidentally altering its behaviour. Another image pair may fit the same interface, while separate measurements for each cilium change the requirements. Saved decisions and checks help a future colleague distinguish these cases.

**Assignment 4h:** Give a brief quality review of the planning artefacts, prompt, checks, and implementation. Include one strength, one limitation, and what a colleague would need to reuse or maintain the result.

**Predict:** Which evidence would be most useful when diagnosing a wrong result later?

**Our quality review:** \<Add your solution here\>

**Reflect:** Where was your judgement essential, and what did the LLM contribute? Do you understand the accepted result well enough to take responsibility for its quality?